# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, utilizing Croissant schema concepts and explicit `@id` referencing for all key dataset components.

### Dataset Source
The dataset source is provided via a Croissant JSON-LD schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Use these IDs for all further referencing.

We enumerate each record set and print its fields and columns by `@id`.

In [ ]:
# List all available record sets and their fields by `@id`
record_sets = dataset.metadata.recordSet
print("Record sets in dataset:")
for rs in record_sets:
    print(f"  Record Set @id: {rs['@id']}")
    if 'field' in rs:
        print("    Fields:")
        for fld in rs['field']:
            fld_id = fld["@id"] if isinstance(fld, dict) and "@id" in fld else str(fld)
            print(f"      - {fld_id}")
    # Show columns if present
    if 'column' in rs:
        print("    Columns:")
        for col in rs['column']:
            col_id = col["@id"] if isinstance(col, dict) and "@id" in col else str(col)
            print(f"      - {col_id}")
    print()

# Show an example record for each record set using the mlcroissant records() generator
for rs in record_sets:
    print(f"Example record for Record Set {rs['@id']}:")
    try:
        for idx, rec in enumerate(dataset.records(record_set=rs['@id'])):
            if idx >= 1: break
            print(rec)
    except Exception as e:
        print(f"  Error extracting records: {e}")
    print('-'*60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use correct record set and field `@id`s listed above.

If there is only one record set, we demonstrate on that set by its exact `@id`.

In [ ]:
# Choose RecordSet by `@id` (fill in from overview above)
# Example for the main protein record set (change @id as needed)
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Extracting records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    if not df.empty:
        dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  [No records or empty set]")
    print()
# For this dataset, pick the main record set for deep dive:
MAIN_RECORD_SET_ID = record_set_ids[0] if record_set_ids else None

# Show columns and first 5 rows
if MAIN_RECORD_SET_ID:
    print(f"Main record set columns: {dataframes[MAIN_RECORD_SET_ID].columns.tolist()}")
    display(dataframes[MAIN_RECORD_SET_ID].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. Always reference columns by their `@id` as in the schema.

In [ ]:
# Select a numeric field for analysis (update with actual @id)
df = dataframes[MAIN_RECORD_SET_ID]

# Example: Find a likely numeric field by @id (printed in previous cell)
# For demonstration, try common proteomics fields such as 'abundance' or 'MW' (Molecular Weight)
# Replace with actual column names as needed
import numpy as np
numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
if not numeric_candidates:
    # Try to cast columns to float that look like numbers
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]

print(f"Numeric field candidates: {numeric_candidates}")

# Pick the first candidate
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field}")

    # Perform a threshold filter (> median)
    threshold = df[numeric_field].median()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Records with {numeric_field} > {threshold} (median): {len(filtered_df)}")

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by another field (e.g. accession or description if present)
    non_numeric = [col for col in df.columns if df[col].dtype == object]
    group_field = non_numeric[0] if non_numeric else None
    if group_field and group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean of {numeric_field} by {group_field} (showing up to 5):")
        print(grouped.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the main numeric field, and a boxplot by a group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field:
        # Boxplot by group
        nuniq = df[group_field].nunique()
        if nuniq > 2 and nuniq < 50:
            plt.figure(figsize=(10,5))
            sns.boxplot(data=df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=90)
            plt.show()
else:
    print('No numeric field available to plot.')

## 6. Conclusion
In this notebook, we've demonstrated how to load and analyze a Croissant-formatted FAIR² dataset using the `mlcroissant` library. All core dataset components (record sets, fields) are referenced by their `@id`. 

- The dataset is well-documented via a Croissant schema, supporting robust and reproducible data science workflows.
- With field- and record-centric loading, users can easily extract, filter, normalize, and visualize protein data.

**Next steps**: Adapt filtering/grouping logic to specific biological or experimental hypotheses. For best accuracy, refer to the dataset's published [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) to align `@id` references with your use case.